# Step 2 — Reliability-Aware Design

This notebook documents the architecture, signals, decision policy, and trace schema for the Reliable Adaptive Agentic RAG system.

It is a **design document** — mostly markdown with a small amount of Python for loading data artifacts.
  
**Date:** 29 May 2026

---

## 2.1 Architecture Overview

The system adds a **reliability layer** on top of the existing multi-agent orchestration (Confidence / Waterfall / Voting).

```
User Query
    │
    ▼
┌─────────────────┐
│ ClarificationAgent│─── needs clarification? ──→ return clarification question
└─────────────────┘
    │ no
    ▼
┌─────────────────┐      ┌─────────────────┐
│  Retrieve docs  │ ────→│ Confidence /    │
│  (strategy-A)   │      │ Waterfall /     │
└─────────────────┘      │ Voting          │
    │                    └─────────────────┘
    ▼
┌─────────────────┐
│ AnswerSynthesizer│─── generates candidate answer from top-k docs
└─────────────────┘
    │
    ▼
┌──────────────────────────────────────────────────────────────┐
│                    RELIABILITY LAYER                         │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐           │
│  │ Sufficiency │  │ Groundedness│  │ Contradict│             │
│  │   Agent     │  │   Agent     │  │   Agent     │           │
│  └─────────────┘  └─────────────┘  └─────────────┘           │
│         │                │                │                  │
│         └────────────────┴────────────────┘                  │
│                          ▼                                   │
│                   ┌─────────────┐                            │
│                   │  TrustAgent │─── combined reliability score│
│                   └─────────────┘                            │
│                          │                                   │
│         ┌────────────────┼────────────────┐                  │
│         ▼                ▼                ▼                  │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐           │
│  │  Abstention │  │   Critic    │  │  Recovery   │           │
│  │   Agent     │  │   Agent     │  │   Agent     │           │
│  └─────────────┘  └─────────────┘  └─────────────┘           │
│         │                │                │                  │
│         └────────────────┴────────────────┘                  │
│                          ▼                                   │
│                   Final Decision: answer / abstain / recover │
└──────────────────────────────────────────────────────────────┘
    │
    ▼
Unified Trace Dict (§2.4)
```

### Key design principles

1. **Reuse legacy code, do not rewrite**: The retrieval pipeline from the old project lives in `multi-agent-step-2_strategy-A.ipynb`. It is loaded as an internal library (via `%run`) in Step 3. It is **not** a deliverable in the current Step 1-4 workflow -- it is background infrastructure we build on top of.
2. **Modular agents**: Each reliability check is a separate agent with a single responsibility.
3. **Interpretable by default**: Every run produces a trace dict (§2.4) that explains every decision.
4. **Graceful degradation**: If reliability is low, the system abstains or recovers — it never silently gives a wrong answer.

5. **Answer synthesis — RESOLVED**: Step 3 now uses the real `AnswerSynthesizerAgent` from `multi-agent-step-2_strategy-A.ipynb`. The orchestrator's `.run()` method returns a synthesized `final_answer`, which Step 3 captures and uses directly. Fallback to `docs[0].page_content[:250]` only occurs if the synthesizer returns empty text.



---

## 2.2 Reliability Signals Catalog

We use **5 reliability signals**. Each is produced by one agent and consumed by the TrustAgent and the decision policy.

| # | Signal | Agent | Description | Range / Type |
|---|--------|-------|-------------|--------------|
| 1 | **Evidence Sufficiency** | `EvidenceSufficiencyAgent` | Measures whether retrieved documents contain enough information to answer the query. Based on query-term coverage in the top-k chunks. | `float` 0.0–1.0 |
| 2 | **Grounding Score** | `GroundednessAgent` | Checks whether the generated answer is supported by the retrieved documents. Based on token overlap between answer and top-k chunks. | `float` 0.0–1.0 |
| 3 | **Has Contradictions** | `ContradictionAgent` | Detects opposing facts in the retrieved documents (e.g. "increases" vs "decreases"). Uses keyword-pair matching. | `bool` |
| 4 | **Query Ambiguous** | `ClarificationAgent` | Detects queries that are too short, contain pronouns, or lack entities. Triggers clarification before retrieval. | `bool` |
| 5 | **Trust Score** | `TrustAgent` | Combined score computed from signals 1–3. Higher = more confident the answer is reliable. | `float` 0.0–1.0 |

### Signal flow

```
Sufficiency ──┐
              ├──→ TrustAgent ──→ Trust Score
Groundedness ─┤          │
              │          ├──→ AbstentionAgent
Contradiction─┘          ├──→ CriticAgent
                         └──→ RecoveryAgent
```

### Why these 5?

- **Sufficiency** catches "the docs don't cover the question" (retrieval failure).
- **Groundedness** catches "the answer makes things up" (synthesis / hallucination failure).
- **Contradiction** catches "the docs disagree" (conflicting evidence).
- **Ambiguity** catches "the user didn't ask a clear question" (before wasting retrieval effort).
- **Trust** is the single scalar that drives the final decision — simple and interpretable.

---

## 2.3 Decision Policy

The policy is a deterministic rule tree with **4 possible outcomes**.

### Pseudocode

```python
def decide(query, docs, answer, signals):
    # 1. CLARIFY branch (pre-retrieval)
    if signals.query_ambiguous:
        return {
            "decision": "clarify",
            "reason": "Query is ambiguous or too short",
            "response": clarification_question(query),
        }

    # 2. ABSTAIN branch (post-retrieval, low trust)
    if signals.trust_score < ABSTAIN_THRESHOLD:
        return {
            "decision": "abstain",
            "reason": "Trust score below threshold",
            "response": "The system cannot answer reliably.",
        }

    # 3. RECOVER branch (post-retrieval, recoverable failure)
    recovery = recovery_agent.suggest(query, strategy, signals)
    if recovery["action"] in {"rewrite", "switch_strategy"}:
        new_docs, new_answer = re_retrieve(query, recovery["action"])
        new_signals = re_evaluate(new_docs, new_answer)
        if new_signals.trust_score >= ABSTAIN_THRESHOLD:
            return {
                "decision": "answer",
                "reason": "Recovery succeeded",
                "response": new_answer,
                "intermediate": {"retry_count": 1, "recovery_action": recovery["action"]},
            }
        else:
            # Recovery failed → escalate to abstain
            return {
                "decision": "abstain",
                "reason": "Recovery attempted but still unreliable",
                "response": "The system cannot answer reliably.",
            }

    # 4. ANSWER branch (default, high trust)
    return {
        "decision": "answer",
        "reason": "All signals above thresholds",
        "response": answer,
    }
```

### Thresholds (configurable)

| Parameter | Default | Meaning |
|-----------|---------|---------|
| `ABSTAIN_THRESHOLD` | 0.4 | Trust score below this → abstain or recover |
| `RECOVERY_MAX_RETRIES` | 1 | Only one recovery attempt per query |

### Branch priority

The order matters: **Clarify → Abstain → Recover → Answer**.

- If the query is ambiguous, we stop immediately (no point retrieving).
- If trust is too low and recovery can't help, we abstain gracefully.
- If recovery can help, we try once and re-evaluate.
- Only if everything looks good do we return the answer.

---

## 2.4 Unified Trace Schema

Every call to `ReliableAdaptiveRAG.run()` returns the **same dict shape** regardless of which branch is taken. This makes logging, debugging, and evaluation uniform.

### Schema definition (plain Python dict)

```python
{
    "decision": str,            # one of: "answer", "clarify", "abstain", "recover"
    "reason": str,              # human-readable explanation of the decision
    "signals": {
        "evidence_sufficiency": float,   # 0.0–1.0
        "grounding_score":      float,   # 0.0–1.0
        "has_contradictions":   bool,    # True / False
        "query_ambiguous":      bool,    # True / False
        "trust_score":          float,   # 0.0–1.0
    },
    "intermediate": {
        "strategy_used": str,           # e.g. "confidence", "waterfall", "voting"
        "recovery_action": str | None,  # "rewrite", "switch_strategy", or None
        "retry_count": int,             # 0 = no retry, 1 = recovered once
    },
    "final_answer": str | None,    # the answer text, or None if clarify/abstain
    "trace_log": [str],          # chronological list of human-readable steps
}
```

### Full example — answer branch

```json
{
    "decision": "answer",
    "reason": "All signals above thresholds",
    "signals": {
        "evidence_sufficiency": 0.85,
        "grounding_score": 0.78,
        "has_contradictions": false,
        "query_ambiguous": false,
        "trust_score": 0.82
    },
    "intermediate": {
        "strategy_used": "confidence",
        "recovery_action": null,
        "retry_count": 0
    },
    "final_answer": "ETH Zurich was founded in 1855 as the Swiss Federal Polytechnic School.",
    "trace_log": [
        "Query received: When was ETH Zurich founded?",
        "Clarification check: not ambiguous",
        "Retrieved 5 docs using strategy=confidence",
        "Answer generated from top-1 doc",
        "Sufficiency=0.85, Grounding=0.78, Contradiction=False",
        "Trust score computed: 0.82",
        "Decision: answer"
    ]
}
```

### Full example — recover branch

```json
{
    "decision": "answer",
    "reason": "Recovery succeeded",
    "signals": {
        "evidence_sufficiency": 0.72,
        "grounding_score": 0.65,
        "has_contradictions": false,
        "query_ambiguous": false,
        "trust_score": 0.68
    },
    "intermediate": {
        "strategy_used": "voting",
        "recovery_action": "switch_strategy",
        "retry_count": 1
    },
    "final_answer": "The main building is located on Rämistrasse 101.",
    "trace_log": [
        "Query received: Where is the main ETH building?",
        "Clarification check: not ambiguous",
        "Retrieved 5 docs using strategy=confidence",
        "Trust score=0.35 (below threshold)",
        "Recovery action: switch_strategy → voting",
        "Re-retrieved 5 docs using strategy=voting",
        "Trust score after retry=0.68",
        "Decision: answer"
    ]
}
```

### Design note

We use a **plain Python dict** (not Pydantic, not dataclass) because:
- It is beginner-friendly — no imports, no validation errors.
- It is flexible — we can add fields later without changing a schema definition.
- It serializes directly to JSON for saving and inspecting.

Production upgrade path (cited in Step 5): migrate to `pydantic.BaseModel` for runtime validation and type hints.

---

## 2.5 Failure → Mechanism Mapping

This section links each failure category from the failure taxonomy (Step 1, §1.8) to the reliability mechanism that addresses it.

### Expected mapping (from spec)

| Failure Category | Mechanism that addresses it | How |
|------------------|------------------------------|-----|
| **Retrieval failure** | `RecoveryAgent` + `EvidenceSufficiencyAgent` | If docs don't cover the query, sufficiency is low → triggers recovery (re-retrieve or switch strategy) |
| **Ranking failure** | `RecoveryAgent` | Switch strategy to get a different ranking (e.g. voting or waterfall) |
| **Synthesis failure** | `GroundednessAgent` + `CriticAgent` | Detects when answer is not supported by docs; critic provides feedback |
| **Grounding failure** | `GroundednessAgent` | Measures token overlap between answer and docs; low score → abstain or recover |
| **Ambiguity failure** | `ClarificationAgent` | Detects short/ambiguous queries before retrieval; asks user for clarification |
| **Contradiction failure** | `ContradictionAgent` | Detects opposing facts in retrieved docs; sets `has_contradictions=True` → lowers trust |
| **Orchestration failure** | `RecoveryAgent` + `TrustAgent` | If different strategies disagree, trust score reflects uncertainty; recovery can switch strategy |
| **Overconfidence failure** | `TrustAgent` + `AbstentionAgent` | High confidence but low metrics → trust score is calibrated downward → system abstains |

### Concrete examples

*(To be filled after Step 1 failure taxonomy is generated. Each row above will get 1–2 real query examples from the benchmark.)*

```markdown
Example (placeholder):
- Failure: retrieval failure
- Query: "What is the admission rate for the D-MAVT program?"
- Observation: Gold passage not in top-10 retrieved docs
- Mechanism triggered: EvidenceSufficiencyAgent scored 0.12 → RecoveryAgent switched to waterfall strategy → docs found on retry
```

In [ ]:
# Optional: load Step 1 failure taxonomy if already available
# If Step 1 is not yet complete, this cell will fail gracefully and we use the placeholder mapping above.

import json
from pathlib import Path

# Adjust PROJECT_ROOT to your Drive path when running on Colab
PROJECT_ROOT = Path(".")  # local fallback

taxonomy_csv = PROJECT_ROOT / "results" / "failure_taxonomy.csv"
examples_json = PROJECT_ROOT / "results" / "failure_examples.json"

if taxonomy_csv.exists():
    import pandas as pd
    df = pd.read_csv(taxonomy_csv)
    print("Failure taxonomy loaded:")
    print(df.head())
else:
    print("Note: failure_taxonomy.csv not found. Step 1 not yet complete.")
    print("Using placeholder mapping from §2.5 above.")

if examples_json.exists():
    with open(examples_json) as f:
        examples = json.load(f)
    print(f"\nLoaded {len(examples)} example categories.")
else:
    print("Note: failure_examples.json not found.")

---

## 2.6 Integration with Legacy Step 2

The reliability layer does **not** replace the existing multi-agent pipeline — it **wraps** it.

### Data flow

```
User Query
    │
    ▼
┌────────────────────────────────────────────┐
│  Legacy Step 2 (strategy-A)                │
│  • QueryUnderstandingAgent                  │
│  • ConfidenceOrchestrator / Waterfall /     │
│    VotingOrchestrator                       │
│  • Fusion → ReRank → AnswerSynthesizer      │
│  → produces: docs + draft_answer + trace    │
└────────────────────────────────────────────┘
    │
    ▼
┌────────────────────────────────────────────┐
│  Step 3 Reliability Layer                   │
│  • Sufficiency / Groundedness / Contradict  │
│  • Trust → Abstention / Critic / Recovery   │
│  → produces: final_decision + unified_trace │
└────────────────────────────────────────────┘
```

### Code reuse

- **Step 3 loads strategy-A** via `%run multi-agent-step-2_strategy-A.ipynb`.
- The orchestrators (`ConfidenceOrchestrator`, `WaterfallOrchestrator`, `VotingOrchestrator`) are reused unchanged.
- The answer synthesizer is reused in Step 3: wrapper functions capture the orchestrator's `final_answer` and pass it to the reliability layer (see §2.1, principle 5).
- All reliability agents are **new** and live in `Step_3_Reliable_Adaptive_Agentic_RAG.ipynb`.

### Why wrap instead of rewrite?

- **Minimal risk**: We keep the proven retrieval pipeline intact.
- **Modular testing**: We can evaluate strategy-A alone (baseline) and with the reliability layer (new system) side-by-side.
- **Clear ablation**: In Step 4, `ReliableAdaptiveRAG(ablate=[...])` disables individual reliability checks while keeping the same retrieval pipeline.



---

## 2.7 Summary

This design document establishes:

1. **Architecture**: A reliability layer wrapping the existing multi-agent orchestration.
2. **Signals**: 5 interpretable signals that feed into a single trust score.
3. **Policy**: A deterministic 4-branch decision tree (clarify → abstain → recover → answer).
4. **Trace schema**: A plain Python dict that every run returns, making the system fully interpretable.
5. **Failure mapping**: Each spec failure category is mapped to a concrete mechanism.


6. **Known limitations & upgrade path**: The current design intentionally uses lightweight heuristics so the system remains understandable and debuggable. Each heuristic has a clear upgrade path to a more rigorous implementation:
   - **Evidence Sufficiency**: currently token-overlap based → future: semantic coverage or LLM-based sufficiency scoring.
   - **Groundedness**: currently token-overlap based → future: NLI (natural language inference) model for entailment checking.
   - **Contradiction**: currently keyword-pair matching (`yes/no`, `increase/decrease`) → future: LLM-based semantic contradiction detection or structured claim extraction + comparison.
   - **Trust scoring**: currently a fixed linear formula (`0.6*sufficiency + 0.3*groundedness - 0.4*contradiction`) → future: calibration on real evaluation data or learned weights.
   - **Clarification**: currently pronoun/length check → future: LLM for entity disambiguation and intent clarification.
   - **Answer synthesis**: currently a placeholder (`docs[0].page_content[:250]`) → future: full `AnswerSynthesizerAgent` from the legacy Step 2 notebook.

The next step (Step 3) implements these mechanisms in code.